# ChEMBL Data Retrieval

In [1]:
from chembl_webresource_client.new_client import new_client

/Users/vlad/Documents/University/Master-MIND/DALAS-Project/.venv/lib/python3.11/site-packages/chembl_webresource_client/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __version__ = __import__('pkg_resources').get_distribution('chembl_webresource_client').version


In [2]:
molecule = new_client.molecule
drug_indication = new_client.drug_indication
target = new_client.target
activity = new_client.activity

In [21]:
# Filter out only autoimmune diseases
autoimmune_ind = drug_indication.filter(mesh_id__exact="D000224")

In [ ]:
import requests
from bs4 import BeautifulSoup
# Exctract all MeSH terms of autoimmune diseases
# Site with MeSH terms with autoimmune diseases
url = "https://www.ncbi.nlm.nih.gov/mesh?Db=mesh&Cmd=DetailsSearch&Term=%22Autoimmune+Diseases%22%5BMeSH+Terms%5D"
response = requests.get(url)
if response.status_code == 200:
    html_content = response.text
    soup = BeautifulSoup(html_content, "html.parser")
else:
    html_content = None


#[entry['ui'] for entry in data['mesh']['descriptorRecord']]


In [ ]:
soup.find_all("span", string="Autoimmune Diseases")

[<span class="highlight" style="background-color:">Autoimmune Diseases</span>,
 <span class="highlight" style="background-color:">Autoimmune Diseases</span>,
 <span class="highlight" style="background-color:">Autoimmune Diseases</span>,
 <span class="highlight" style="background-color:">Autoimmune Diseases</span>]

In [53]:
autoimm_ul = soup.find("span", string="Autoimmune Diseases").find_all_next("ul")

In [65]:
autoimm_ul[8].find_all("a")[0].get("href")

'/mesh/68000224'

In [79]:
mesh_ids = []

for disease_a in autoimm_ul[8].find_all("a"):
    disease_url = disease_a.get('href')
    response = requests.get(f'https://www.ncbi.nlm.nih.gov{disease_url}')
    if response.status_code == 200:
        html_content = response.text
        disease_soup = BeautifulSoup(html_content, "html.parser")
        mesh_ids.append(disease_soup.find("p", string=lambda text: text and text.startswith("MeSH Unique ID:")).text.split()[-1])

In [86]:
len(mesh_ids)

50

In [87]:
with open("mesh_ids.txt", "w") as f:
    f.writelines(f"{_id}\n" for _id in mesh_ids)

In [89]:
with open("mesh_ids.txt", "r") as f:
    lines = f.readlines()

mesh_ids = [line.strip() for line in lines]
autoimmune_ind = drug_indication.filter(mesh_id__in=mesh_ids)

In [90]:
len(autoimmune_ind)

1256

In [92]:
autoimmune_ind[10]

{'drugind_id': 23145,
 'efo_id': 'EFO:0002609',
 'efo_term': 'juvenile idiopathic arthritis',
 'indication_refs': [{'ref_id': 'NCT00078806,NCT00443430,NCT00962741,NCT01287715,NCT01421069,NCT02840175,NCT03780959,NCT03781375',
   'ref_type': 'ClinicalTrials',
   'ref_url': 'https://clinicaltrials.gov/search?term=NCT00078806%20NCT00443430%20NCT00962741%20NCT01287715%20NCT01421069%20NCT02840175%20NCT03780959%20NCT03781375'},
  {'ref_id': 'a002b40c-097d-47a5-957f-7a7b1807af7f',
   'ref_type': 'DailyMed',
   'ref_url': 'https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid=a002b40c-097d-47a5-957f-7a7b1807af7f'},
  {'ref_id': 'EMEA/H/C/000262',
   'ref_type': 'EMA',
   'ref_url': 'https://www.ema.europa.eu/en/medicines/human/EPAR/enbrel'}],
 'max_phase_for_ind': '4.0',
 'mesh_heading': 'Arthritis, Juvenile',
 'mesh_id': 'D001171',
 'molecule_chembl_id': 'CHEMBL1201572',
 'parent_molecule_chembl_id': 'CHEMBL1201572'}